In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEXT_COL  = "text"
LABEL_COL = "label"

# 1) Load
ds = load_dataset("yelp_review_full")

# 2) Extract to plain Python / NumPy containers (no more HF indexing later)
#    Use list(...) to force a real Python list (not HF's internal mapping),
#    and np.asarray(..., dtype=np.int64) for labels.
X_all_list  = list(ds["train"][TEXT_COL])                         # list[str]
y_all_arr   = np.asarray(ds["train"][LABEL_COL], dtype=np.int64)  # np.ndarray[int64]
X_test_list = list(ds["test"][TEXT_COL])                          # list[str]
y_test_arr  = np.asarray(ds["test"][LABEL_COL], dtype=np.int64)   # np.ndarray[int64]

# (Optional but recommended) prevent accidental re-use of HF dataset
del ds

# 3) Split using scikit-learn — returns lists/ndarrays; no indices back into HF
X_train_list, X_val_list, y_train_arr, y_val_arr = train_test_split(
    X_all_list, y_all_arr, test_size=0.10, stratify=y_all_arr, random_state=RANDOM_STATE
)

# 4) (Safety) ensure lists are *really* lists of str, and labels are pure NumPy arrays
X_train_list = list(X_train_list)
X_val_list   = list(X_val_list)
X_test_list  = list(X_test_list)

y_train_arr  = np.asarray(y_train_arr, dtype=np.int64)
y_val_arr    = np.asarray(y_val_arr, dtype=np.int64)
y_test_arr   = np.asarray(y_test_arr, dtype=np.int64)

print(
    "Sizes →",
    "Train:", len(X_train_list),
    "Val:",   len(X_val_list),
    "Test:",  len(X_test_list)
)
print("Label ranges →",
      "train:", (int(y_train_arr.min()), int(y_train_arr.max())),
      "val:",   (int(y_val_arr.min()),   int(y_val_arr.max())),
      "test:",  (int(y_test_arr.min()),  int(y_test_arr.max())))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Sizes → Train: 585000 Val: 65000 Test: 50000
Label ranges → train: (0, 4) val: (0, 4) test: (0, 4)


In [2]:
# 1) Vectorize (fit ONLY on train)
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TFIDF_KW = dict(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50_000,   # if you hit RAM issues, try 30_000 or 20_000
    min_df=5,
    max_df=0.95,
    dtype=np.float32,      # saves memory
)

tfidf = TfidfVectorizer(**TFIDF_KW)
Xtr = tfidf.fit_transform(X_train_list)
Xva = tfidf.transform(X_val_list)
Xte = tfidf.transform(X_test_list)

Xtr.shape, Xva.shape, Xte.shape


KeyboardInterrupt: 

In [ ]:
# 2) Handle class imbalance with sample weights
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_arr)
class_weights = compute_class_weight("balanced", classes=classes, y=y_train_arr)
cw_map = {c:w for c,w in zip(classes, class_weights)}
sample_w = np.array([cw_map[c] for c in y_train_arr], dtype=np.float32)
cw_map


In [ ]:
# 3) Train XGBoost on GPU (XGBoost ≥ 2.x)
from xgboost import XGBClassifier

clf = XGBClassifier(
    objective="multi:softprob",
    num_class=int(classes.max() + 1),  # 5 for Yelp
    n_estimators=2000,                 # ES will stop early
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method="hist",                # with device="cuda" for GPU on 2.x
    device="cuda",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42,
)

print("Training (device=cuda, tree_method=hist) with early stopping…")
clf.fit(
    Xtr, y_train_arr,
    sample_weight=sample_w,
    eval_set=[(Xva, y_val_arr)],
    early_stopping_rounds=50,
    verbose=100,
)


In [ ]:
# 4) Evaluate
from sklearn.metrics import classification_report, accuracy_score

LABEL_NAMES = ["1 star","2 stars","3 stars","4 stars","5 stars"]

y_val_pred = clf.predict(Xva)
print("Val accuracy:", accuracy_score(y_val_arr, y_val_pred))
print(classification_report(y_val_arr, y_val_pred, target_names=LABEL_NAMES, digits=4))

y_test_pred = clf.predict(Xte)
print("Test accuracy:", accuracy_score(y_test_arr, y_test_pred))
print(classification_report(y_test_arr, y_test_pred, target_names=LABEL_NAMES, digits=4))
